# k-nearest neighbors
**Machine Learning · Week 1 (Learning from examples) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w1d3.html

To classify a new point, find the *k* closest labeled points and let them vote.
First we write it ourselves in a few lines, then use scikit-learn and draw decision boundaries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.25, random_state=0)
print("X:", X.shape, " y:", y.shape, " classes:", np.unique(y))

plt.scatter(X[:, 0], X[:, 1], c=np.where(y == 0, "#ff8a50", "#5577f5"), s=30)
plt.title("Two moons: our toy dataset")
plt.show()

## kNN from scratch
1. measure the distance from the new point to every training point,
2. take the `k` smallest,
3. return the most common label among them.

In [ ]:
def knn_predict(X_train, y_train, x_new, k=5):
    distances = np.sqrt(((X_train - x_new) ** 2).sum(axis=1))   # Euclidean distance
    nearest = np.argsort(distances)[:k]                          # indices of the k closest
    votes = np.bincount(y_train[nearest], minlength=2)           # count labels
    return votes.argmax(), votes

x_new = np.array([0.5, 0.25])
for k in [1, 5, 25]:
    label, votes = knn_predict(X, y, x_new, k)
    print(f"k = {k:2d}: votes {votes} -> class {label}")

Notice that the answer can change with `k`. Small `k` listens to the very closest points;
large `k` asks a crowd.

## The same with scikit-learn, and the decision boundary
Coloring every point of the plane by the predicted class shows the **decision boundary**:
the line where the vote flips.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from matplotlib.colors import LinearSegmentedColormap

apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])
xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 220))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True, layout="constrained")
for ax, k in zip(axes, [1, 15, 75]):
    model = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    zz = model.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=20, cmap=apricot_blue, alpha=0.45)
    ax.contour(xx, yy, zz, levels=[0.5], colors="#2c2842", linewidths=1.5)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 0, "#ff8a50", "#5577f5"), s=14)
    ax.set_title(f"k = {k}")
plt.show()

## Your turn: kNN for many points at once, in PyTorch
Fill in the three lines of `knn_torch`. It should classify **all** rows of `X_new` at once.

In [ ]:
import torch

def knn_torch(X_train, y_train, X_new, k=5):
    distances = None   # TODO: torch.cdist(...), shape (n_new, n_train)
    indices = None     # TODO: the k smallest distances in each row: .topk(k, largest=False).indices
    return None        # TODO: the most common neighbor label in each row: y_train[indices].mode(dim=1).values

Xt, yt = torch.tensor(X, dtype=torch.float32), torch.tensor(y)
X_new = torch.tensor(grid[::97], dtype=torch.float32)       # about 680 new points spread over the plane
pred = knn_torch(Xt, yt, X_new, k=15)
if pred is None:
    print("knn_torch returns None so far: fill in the TODOs")
else:
    sk = KNeighborsClassifier(n_neighbors=15).fit(X, y).predict(X_new.numpy())
    print(f"agrees with scikit-learn on {(pred.numpy() == sk).mean():.1%} of {len(X_new)} points")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def knn_torch(X_train, y_train, X_new, k=5):
    distances = torch.cdist(X_new, X_train)                 # every new point to every training point
    indices = distances.topk(k, largest=False).indices      # the k nearest, per row
    return y_train[indices].mode(dim=1).values              # majority vote, per row

pred = knn_torch(Xt, yt, X_new, k=15)
sk = KNeighborsClassifier(n_neighbors=15).fit(X, y).predict(X_new.numpy())
print(f"agrees with scikit-learn on {(pred.numpy() == sk).mean():.1%} of {len(X_new)} points")
print("predicted classes for the first 10 new points:", pred[:10].tolist())